# Gold 3_14 — Customer Engagement Versions (SCD2)

One row per lead per **engagement version** — a new version opens only when the engagement hash changes. Complements 3_15 (daily snapshots = trends) with a true change log (versions = what changed, and when).

* **Population**: all Coaching Client leads from `gold.health_metrics` (intern clients included — currently 1,371).
* **Hash** (opens a version on change): LEAD_ID, LEAD_EMAIL, LEAD_NAME, CONTRACT_VALUE, SENTIMENTS_LAST_30_DAYS, SLACK_ENGAGED_FLAG, PLATFORM_ENGAGED_FLAG, MEETING_ENGAGED_FLAG, ENGAGEMENT_PATTERN, BASE_ENGAGEMENT_SCORE, FINAL_SCORE, HEALTH_BAND.
* **VRSN_START_DATE** = run date the version opened (first run = today). **VRSN_END_DATE** = run date the hash changed (the version was superseded); NULL while current.
* Clock metrics (DAYS_SINCE_*, TOTAL_MISSED_CALLS, CONTRACT_START_DATE) are **not hashed** — they refresh on the open version every run; closed versions keep their as-of values.
* TOTAL_MISSED_CALLS is the trailing **14-day window** count, not a lifetime total. It still influences versions through FINAL_SCORE / HEALTH_BAND; only score-neutral call changes (e.g. 1→2) do not open one.
* A lead that leaves the Coaching Client population gets its open version closed that run; if it returns, a new version opens.
* SOURCE_UPDATED_AT = the CRM lead record's `date_updated`. INSERT_DATE = when this pipeline created the version row.
* Scoring identical to 3_14 (spec 5.1): base from pattern (90, 80, 75, 60, 50, 40, 30, 20, or no-data), FINAL_SCORE = base + missed-call adjustment + sentiment adjustment — no adjustments on NO DATA, not capped.

In [0]:
%sql
-- Engagement source: the 3_14 scoring logic applied to the FULL health_metrics
-- population (intern clients included — this is the versioned record of everyone),
-- plus the change-detection hash. Hash = engagement state + identity + contract
-- value; clock metrics deliberately excluded (they drift daily and would
-- degenerate SCD2 into a daily snapshot — 3_15 already owns that job).
CREATE OR REPLACE TEMP VIEW customer_engagement_source AS (
WITH base AS (
    SELECT
        h.lead_id                                                          AS LEAD_ID,
        h.lead_name                                                        AS LEAD_NAME,
        h.primary_customer_email                                           AS LEAD_EMAIL,
        TRY_CAST(REGEXP_REPLACE(h.Contracted_Value, '[^0-9.]', '') AS DOUBLE) AS CONTRACT_VALUE,
        -- Contract start: CRM custom field first (reference source, 1,323/1,371
        -- clients), sale-activity date as fallback for leads the field misses.
        COALESCE(TRY_CAST(h.contract_start_date AS DATE),
                 TRY_CAST(h.date_of_sale AS DATE))                  AS CONTRACT_START_DATE,
        h.days_since_last_email                                            AS DAYS_SINCE_LAST_EMAIL,
        h.days_since_last_login                                            AS DAYS_SINCE_LAST_LOGIN,
        h.days_since_last_meeting                                          AS DAYS_SINCE_LAST_MEETING,
        h.days_since_last_message_from_client                              AS DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT,
        h.days_since_last_message_from_team_member                         AS DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
        h.missed_calls_14_days                                             AS TOTAL_MISSED_CALLS,
        COALESCE(UPPER(TRIM(h.sentiment)), 'NO_SENTIMENT')                 AS SENTIMENTS_LAST_30_DAYS,
        TRY_CAST(h.date_updated AS TIMESTAMP)                              AS SOURCE_UPDATED_AT,
        CASE WHEN h.days_since_last_message_from_client IS NULL THEN NULL
             WHEN h.days_since_last_message_from_client <= 14 THEN 1
             ELSE 0 END                                                    AS SLACK_ENGAGED_FLAG,
        CASE WHEN h.days_since_last_login IS NULL THEN NULL
             WHEN h.days_since_last_login <= 14 THEN 1
             ELSE 0 END                                                    AS PLATFORM_ENGAGED_FLAG,
        CASE WHEN h.days_since_last_meeting IS NULL OR h.days_since_last_meeting_calendly IS NULL THEN NULL
             WHEN h.days_since_last_meeting <= 30 OR h.days_since_last_meeting_calendly <= 30 THEN 1
             ELSE 0 END                                                    AS MEETING_ENGAGED_FLAG,
        CASE WHEN h.missed_calls_14_days = 0 THEN 5
             WHEN h.missed_calls_14_days BETWEEN 1 AND 2 THEN 0
             WHEN h.missed_calls_14_days >= 3 THEN -15 END                 AS missed_call_adjustment,
        CASE WHEN UPPER(TRIM(h.sentiment)) = 'POSITIVE' THEN 10
             WHEN UPPER(TRIM(h.sentiment)) = 'NEUTRAL'  THEN 5
             WHEN UPPER(TRIM(h.sentiment)) = 'NEGATIVE' THEN -25
             ELSE 0 END                                                    AS sentiment_adjustment
    FROM crm_ingestion.gold.health_metrics h
),
patterns AS (
    SELECT
        b.*,
        CASE
            WHEN b.SLACK_ENGAGED_FLAG = 1 AND b.PLATFORM_ENGAGED_FLAG = 1 AND b.MEETING_ENGAGED_FLAG = 1 THEN 'SLACK + PLATFORM + MEETING'
            WHEN b.SLACK_ENGAGED_FLAG = 1 AND b.PLATFORM_ENGAGED_FLAG = 1 AND (b.MEETING_ENGAGED_FLAG = 0 OR b.MEETING_ENGAGED_FLAG IS NULL) THEN 'SLACK + PLATFORM'
            WHEN (b.SLACK_ENGAGED_FLAG = 0 OR b.SLACK_ENGAGED_FLAG IS NULL) AND b.PLATFORM_ENGAGED_FLAG = 1 AND b.MEETING_ENGAGED_FLAG = 1 THEN 'PLATFORM + MEETING'
            WHEN b.SLACK_ENGAGED_FLAG = 1 AND (b.PLATFORM_ENGAGED_FLAG = 0 OR b.PLATFORM_ENGAGED_FLAG IS NULL) AND b.MEETING_ENGAGED_FLAG = 1 THEN 'SLACK + MEETING'
            WHEN b.SLACK_ENGAGED_FLAG = 1 AND (b.PLATFORM_ENGAGED_FLAG = 0 OR b.PLATFORM_ENGAGED_FLAG IS NULL) AND (b.MEETING_ENGAGED_FLAG = 0 OR b.MEETING_ENGAGED_FLAG IS NULL) THEN 'SLACK ONLY'
            WHEN (b.SLACK_ENGAGED_FLAG = 0 OR b.SLACK_ENGAGED_FLAG IS NULL) AND b.PLATFORM_ENGAGED_FLAG = 1 AND (b.MEETING_ENGAGED_FLAG = 0 OR b.MEETING_ENGAGED_FLAG IS NULL) THEN 'PLATFORM ONLY'
            WHEN (b.SLACK_ENGAGED_FLAG = 0 OR b.SLACK_ENGAGED_FLAG IS NULL) AND (b.PLATFORM_ENGAGED_FLAG = 0 OR b.PLATFORM_ENGAGED_FLAG IS NULL) AND b.MEETING_ENGAGED_FLAG = 1 THEN 'MEETING ONLY'
            WHEN b.SLACK_ENGAGED_FLAG = 0 AND b.PLATFORM_ENGAGED_FLAG = 0 AND b.MEETING_ENGAGED_FLAG = 0 THEN 'NO ENGAGEMENT'
            WHEN b.SLACK_ENGAGED_FLAG = 0 AND b.PLATFORM_ENGAGED_FLAG = 0 AND b.MEETING_ENGAGED_FLAG IS NULL THEN 'NO ENGAGEMENT'
            WHEN b.SLACK_ENGAGED_FLAG IS NULL AND b.PLATFORM_ENGAGED_FLAG = 0 AND b.MEETING_ENGAGED_FLAG = 0 THEN 'NO ENGAGEMENT'
            WHEN b.SLACK_ENGAGED_FLAG = 0 AND b.PLATFORM_ENGAGED_FLAG IS NULL AND b.MEETING_ENGAGED_FLAG = 0 THEN 'NO ENGAGEMENT'
            ELSE 'NO DATA'
        END AS ENGAGEMENT_PATTERN
    FROM base b
),
scored AS (
    SELECT
        p.*,
        CASE p.ENGAGEMENT_PATTERN
            WHEN 'SLACK + PLATFORM + MEETING' THEN 90
            WHEN 'SLACK + PLATFORM'         THEN 80
            WHEN 'PLATFORM + MEETING'       THEN 75
            WHEN 'SLACK + MEETING'          THEN 60
            WHEN 'PLATFORM ONLY'            THEN 50
            WHEN 'MEETING ONLY'            THEN 40
            WHEN 'SLACK ONLY'              THEN 30
            WHEN 'NO ENGAGEMENT'           THEN 20
            ELSE NULL
        END AS BASE_ENGAGEMENT_SCORE
    FROM patterns p
),
finals AS (
    SELECT
        s.*,
        CASE WHEN s.ENGAGEMENT_PATTERN = 'NO DATA' THEN NULL
             ELSE s.BASE_ENGAGEMENT_SCORE + s.missed_call_adjustment + s.sentiment_adjustment
        END AS FINAL_SCORE,
        CASE WHEN s.ENGAGEMENT_PATTERN = 'NO DATA' THEN 'NO DATA'
             WHEN s.BASE_ENGAGEMENT_SCORE + s.missed_call_adjustment + s.sentiment_adjustment >= 75 THEN 'GOOD'
             WHEN s.BASE_ENGAGEMENT_SCORE + s.missed_call_adjustment + s.sentiment_adjustment >= 50 THEN 'AVERAGE'
             ELSE 'POOR'
        END AS HEALTH_BAND
    FROM scored s
)
SELECT
    f.LEAD_ID, f.LEAD_EMAIL, f.LEAD_NAME,
    f.CONTRACT_VALUE, f.CONTRACT_START_DATE,
    f.DAYS_SINCE_LAST_EMAIL, f.DAYS_SINCE_LAST_LOGIN, f.DAYS_SINCE_LAST_MEETING,
    f.DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT, f.DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
    f.TOTAL_MISSED_CALLS, f.SENTIMENTS_LAST_30_DAYS,
    f.SLACK_ENGAGED_FLAG, f.PLATFORM_ENGAGED_FLAG, f.MEETING_ENGAGED_FLAG,
    f.ENGAGEMENT_PATTERN, f.BASE_ENGAGEMENT_SCORE, f.FINAL_SCORE, f.HEALTH_BAND,
    f.SOURCE_UPDATED_AT,
    MD5(TO_JSON(STRUCT(
        f.LEAD_ID, f.LEAD_EMAIL, f.LEAD_NAME,
        f.CONTRACT_VALUE, f.SENTIMENTS_LAST_30_DAYS,
        f.SLACK_ENGAGED_FLAG, f.PLATFORM_ENGAGED_FLAG, f.MEETING_ENGAGED_FLAG,
        f.ENGAGEMENT_PATTERN, f.BASE_ENGAGEMENT_SCORE, f.FINAL_SCORE, f.HEALTH_BAND
    ))) AS HASH_VALUE
FROM finals f
)

In [0]:
%sql
-- Bootstrap: create the SCD2 target only if missing. On nightly job runs this is
-- a no-op — the three SCD2 steps below do all the work.
CREATE TABLE IF NOT EXISTS crm_ingestion.gold.customer_health_profile (
    LEAD_ID                             STRING NOT NULL,
    LEAD_EMAIL                          STRING,
    LEAD_NAME                           STRING,
    VRSN_START_DATE                     DATE    COMMENT 'Run date this version opened (first run = first appearance).',
    VRSN_END_DATE                       DATE    COMMENT 'Run date the hash changed and this version was superseded. NULL = current version.',
    CONTRACT_VALUE                      DOUBLE,
    CONTRACT_START_DATE                 DATE,
    DAYS_SINCE_LAST_EMAIL               INT,
    DAYS_SINCE_LAST_LOGIN               INT,
    DAYS_SINCE_LAST_MEETING             INT,
    DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT      BIGINT,
    DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER BIGINT,
    TOTAL_MISSED_CALLS                  BIGINT  COMMENT 'Missed inbound calls in the trailing 14-day window (not a lifetime total).',
    SENTIMENTS_LAST_30_DAYS             STRING,
    SLACK_ENGAGED_FLAG                  INT,
    PLATFORM_ENGAGED_FLAG               INT,
    MEETING_ENGAGED_FLAG                INT,
    ENGAGEMENT_PATTERN                  STRING,
    BASE_ENGAGEMENT_SCORE               INT     COMMENT 'From the pattern table (90, 80, 75, 60, 50, 40, 30, 20, or no-data).',
    FINAL_SCORE                        INT     COMMENT 'Base + missed-call adjustment + sentiment adjustment. No adjustments on NO DATA. Not capped.',
    HEALTH_BAND                         STRING,
    HASH_VALUE                          STRING  COMMENT 'MD5 over the 13 engagement-hash columns; a change opens a new version.',
    SOURCE_UPDATED_AT                   TIMESTAMP COMMENT 'CRM date_updated of the lead record when this version was created.',
    INSERT_DATE                         TIMESTAMP COMMENT 'When this pipeline created the version row.'
);

COMMENT ON TABLE crm_ingestion.gold.customer_health_profile IS
    'SCD2 engagement history: one row per lead per engagement version. A new version opens when the engagement hash changes; VRSN_END_DATE NULL = current. Companion to customer_health_history (daily snapshots).';

In [0]:
%sql
-- SCD2 step 1: close versions whose hash changed, refresh clock metrics on
-- unchanged open versions. Delta evaluates the MATCHED clauses in order.
MERGE INTO crm_ingestion.gold.customer_health_profile AS t
USING customer_engagement_source AS s
ON  t.LEAD_ID = s.LEAD_ID
AND t.VRSN_END_DATE IS NULL
WHEN MATCHED AND t.HASH_VALUE <> s.HASH_VALUE THEN
    UPDATE SET t.VRSN_END_DATE = CURRENT_DATE()
WHEN MATCHED THEN
    UPDATE SET
        t.CONTRACT_START_DATE                 = s.CONTRACT_START_DATE,
        t.DAYS_SINCE_LAST_EMAIL               = s.DAYS_SINCE_LAST_EMAIL,
        t.DAYS_SINCE_LAST_LOGIN               = s.DAYS_SINCE_LAST_LOGIN,
        t.DAYS_SINCE_LAST_MEETING             = s.DAYS_SINCE_LAST_MEETING,
        t.DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT = s.DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT,
        t.DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER = s.DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
        t.TOTAL_MISSED_CALLS                  = s.TOTAL_MISSED_CALLS,
        t.SOURCE_UPDATED_AT                   = s.SOURCE_UPDATED_AT

In [0]:
%sql
-- SCD2 step 2: open a new version for new leads and for leads whose version
-- step 1 just closed. Unchanged leads still have an open version -> skipped.
INSERT INTO crm_ingestion.gold.customer_health_profile (
    LEAD_ID, LEAD_EMAIL, LEAD_NAME, VRSN_START_DATE, VRSN_END_DATE,
    CONTRACT_VALUE, CONTRACT_START_DATE,
    DAYS_SINCE_LAST_EMAIL, DAYS_SINCE_LAST_LOGIN, DAYS_SINCE_LAST_MEETING,
    DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT, DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
    TOTAL_MISSED_CALLS, SENTIMENTS_LAST_30_DAYS,
    SLACK_ENGAGED_FLAG, PLATFORM_ENGAGED_FLAG, MEETING_ENGAGED_FLAG,
    ENGAGEMENT_PATTERN, BASE_ENGAGEMENT_SCORE, FINAL_SCORE, HEALTH_BAND,
    HASH_VALUE, SOURCE_UPDATED_AT, INSERT_DATE
)
SELECT
    s.LEAD_ID, s.LEAD_EMAIL, s.LEAD_NAME,
    CURRENT_DATE()                    AS VRSN_START_DATE,
    CAST(NULL AS DATE)               AS VRSN_END_DATE,
    s.CONTRACT_VALUE, s.CONTRACT_START_DATE,
    s.DAYS_SINCE_LAST_EMAIL, s.DAYS_SINCE_LAST_LOGIN, s.DAYS_SINCE_LAST_MEETING,
    s.DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT, s.DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
    s.TOTAL_MISSED_CALLS, s.SENTIMENTS_LAST_30_DAYS,
    s.SLACK_ENGAGED_FLAG, s.PLATFORM_ENGAGED_FLAG, s.MEETING_ENGAGED_FLAG,
    s.ENGAGEMENT_PATTERN, s.BASE_ENGAGEMENT_SCORE, s.FINAL_SCORE, s.HEALTH_BAND,
    s.HASH_VALUE, s.SOURCE_UPDATED_AT,
    CURRENT_TIMESTAMP()               AS INSERT_DATE
FROM customer_engagement_source s
WHERE NOT EXISTS (
    SELECT 1
    FROM crm_ingestion.gold.customer_health_profile v
    WHERE v.LEAD_ID = s.LEAD_ID
      AND v.VRSN_END_DATE IS NULL
)

In [0]:
%sql
-- SCD2 step 3: a lead that left the Coaching Client population closes its open
-- version today. If it returns later, step 2 opens a fresh version.
UPDATE crm_ingestion.gold.customer_health_profile
SET VRSN_END_DATE = CURRENT_DATE()
WHERE VRSN_END_DATE IS NULL
  AND LEAD_ID NOT IN (SELECT LEAD_ID FROM customer_engagement_source)

In [0]:
%sql
-- Validation 1: open versions must equal the current population (1,339),
-- and every open version started on the first run date.
SELECT
    COUNT(*)            AS open_versions,
    COUNT(DISTINCT LEAD_ID) AS distinct_leads,
    MIN(VRSN_START_DATE)    AS first_version_date,
    MAX(VRSN_START_DATE)    AS latest_version_start
FROM crm_ingestion.gold.customer_health_profile
WHERE VRSN_END_DATE IS NULL

In [0]:
%sql
-- Validation 2: current engagement mix by band (open versions only)
SELECT HEALTH_BAND, ENGAGEMENT_PATTERN, COUNT(*) AS leads
FROM crm_ingestion.gold.customer_health_profile
WHERE VRSN_END_DATE IS NULL
GROUP BY 1, 2
ORDER BY leads DESC

In [0]:
%sql
-- Validation 3: contract start coverage. Source = CRM custom field first
-- (reference output), sale-activity date fallback. Was 154 activity-only before.
SELECT
    COUNT(*)                                              AS open_versions,
    SUM(CASE WHEN CONTRACT_START_DATE IS NOT NULL THEN 1 ELSE 0 END) AS with_contract_start,
    SUM(CASE WHEN CONTRACT_START_DATE IS NULL THEN 1 ELSE 0 END)     AS missing
FROM crm_ingestion.gold.customer_health_profile
WHERE VRSN_END_DATE IS NULL

In [0]:
%sql
-- Validation 4: spot-check traced reference leads (teacher: 2024-12-26 / 2025-10-28 / 2026-06-13)
SELECT LEAD_ID, CONTRACT_START_DATE, CONTRACT_VALUE
FROM crm_ingestion.gold.customer_health_profile
WHERE VRSN_END_DATE IS NULL
  AND LEAD_ID IN ('lead_bo0Aw17YkLn795YW2j2XHTsh43KqkhqjFMjDG1UwH38',
                  'lead_DBOhRgpYGb8kIQXypKsKdxxTy97QY0kJ8CBYsmmL4SG',
                  'lead_9ppVFUPLYDFu7DboruWWUaqWDMq7piyhWxS4enw5oxy')

In [0]:
%sql
-- Validation 3: version churn — leads with more than one version. Empty on the
-- first run; from tomorrow on, this is the change log.
SELECT LEAD_ID, COUNT(*) AS versions, COUNT(*) - 1 AS changes,
       MIN(VRSN_START_DATE) AS first_seen, MAX(VRSN_START_DATE) AS last_change
FROM crm_ingestion.gold.customer_health_profile
GROUP BY LEAD_ID
HAVING COUNT(*) > 1
ORDER BY changes DESC, last_change DESC
LIMIT 10

In [0]:
%sql
select count(distinct lead_id) from crm_ingestion.gold.customer_health_profile

In [0]:
%sql
SELECT
    *
FROM
    crm_ingestion.gold.customer_health_profile
LIMIT 
    100;

In [0]:
%sql
-- VALIDATION — what changed today: for every lead whose version closed today,
-- compare the outgoing (o) vs incoming (n) version. This is the change log behind
-- the version churn validation above, with focus on what the 3_09 login fix moved.
SELECT
    COUNT(*)                                              AS versions_opened_today,
    SUM(CASE WHEN o.PLATFORM_ENGAGED_FLAG IS NULL AND n.PLATFORM_ENGAGED_FLAG IS NOT NULL THEN 1 ELSE 0 END) AS platform_gained_value,
    SUM(CASE WHEN o.PLATFORM_ENGAGED_FLAG IS NULL AND n.PLATFORM_ENGAGED_FLAG = 1 THEN 1 ELSE 0 END) AS platform_gained_engaged,
    SUM(CASE WHEN o.PLATFORM_ENGAGED_FLAG IS NULL AND n.PLATFORM_ENGAGED_FLAG = 0 THEN 1 ELSE 0 END) AS platform_gained_disengaged,
    SUM(CASE WHEN o.PLATFORM_ENGAGED_FLAG = 1 AND n.PLATFORM_ENGAGED_FLAG = 0 THEN 1 ELSE 0 END) AS platform_aged_out,
    SUM(CASE WHEN o.ENGAGEMENT_PATTERN <> n.ENGAGEMENT_PATTERN THEN 1 ELSE 0 END) AS pattern_changed,
    SUM(CASE WHEN o.ENGAGEMENT_PATTERN = 'NO DATA' AND n.ENGAGEMENT_PATTERN <> 'NO DATA' THEN 1 ELSE 0 END) AS left_no_data
FROM crm_ingestion.gold.customer_health_profile n
JOIN crm_ingestion.gold.customer_health_profile o
  ON  o.LEAD_ID = n.LEAD_ID
 AND o.VRSN_END_DATE = CURRENT_DATE()
 AND n.VRSN_START_DATE = CURRENT_DATE()

In [0]:
LEAD_ID,LEAD_EMAIL,LEAD_NAME,VRSN_START_DATE,VRSN_END_DATE,CONTRACT_VALUE,CONTRACT_START_DATE,DAYS_SINCE_LAST_EMAIL,DAYS_SINCE_LAST_LOGIN,DAYS_SINCE_LAST_MEETING,DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT,DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,TOTAL_MISSED_CALLS,SENTIMENTS_LAST_30_DAYS,SLACK_ENGAGED_FLAG,PLATFORM_ENGAGED_FLAG,MEETING_ENGAGED_FLAG,ENGAGEMENT_PATTERN,BASE_ENGAGEMENT_SCORE,FINAL_SCORE,HEALTH_BAND,HASH_VALUE,SOURCE_UPDATED_AT,INSERT_DATE
lead_bo0Aw17YkLn795YW2j2XHTsh43KqkhqjFMjDG1UwH38,sydibuari@yahoo.com / sbuari88@gmail.com,Sydi Buari,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2024-12-26,363,623,370,330,213,0," Positive  ",0,0,,NO DATA,9999,9999,NO DATA,c847cd44e3f4f3087bc12d9cd102e173,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_NEAsiuAhEAuKTWYXmB2c9EEcf7wLlSqRSjMKNX3xdGH,oyegunwaseyi@gmail.com,Seyi Oyegunwa,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,20000,2026-01-27,177,142,50,83,110,0," Positive  ",0,0,0,NO ENGAGEMENT,20,35,POOR,45570028e917ff4db7185bf4b00de031,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_C7o1N9SfcGAsLCqiHGxxeOmVoWQN7ShD9LzATOWcdcm,irron.williams@gmail.com,Irron Williams,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,7000,2025-05-30,375,231,343,259,221,0," Positive  ",0,0,,NO DATA,9999,9999,NO DATA,3fb3932d75a87a85dd67bb498ea894bf,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_Pvq800BVsA671hrI7S6hA4fbAD2dUOeWYJTjL4P8e9n,annambasit123@gmail.com,Annam Basit,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2025-01-11,529,330,445,333,12,0," Positive  ",0,0,,NO DATA,9999,9999,NO DATA,e2a9f37036151d4602266be5ffb923bd,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_9ppVFUPLYDFu7DboruWWUaqWDMq7piyhWxS4enw5oxy,suryadevarabasaveswaramukesh@gmail.com,suryadevarabasaveswaramukesh@gmail.com / Basaveswara Mukesh Suryadevara / Mukesh Suryadevara,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,8000,2026-06-13,33,14,29,9,0,0," Positive  ",1,1,1,SLACK + PLATFORM + MEETING,90,105,GOOD,49002d77f222db86f78033bb1367632a,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_8kvThfspxJJBRfsNxOYhVCuxsQWSZu8CJEF3wLWCX76,nicholo.neri@gmail.com,Nic Neri,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,16000,2025-11-20,230,135,19,215,21,0," No Student Activity  ",0,0,1,MEETING ONLY,40,45,POOR,dc5eaf6196e4405bbbd9f258b9a3856c,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_DBOhRgpYGb8kIQXypKsKdxxTy97QY0kJ8CBYsmmL4SG,attribution.matters@gmail.com,Don / Attribution.matters@gmail.com Attribution.matters@gmail.com,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,19000,2025-10-28,26,2,2,1,2,0," Neutral  ",1,1,1,SLACK + PLATFORM + MEETING,90,100,GOOD,10df54f90b7e09c4f409ed683aa5d79a,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_pAq25PJR446RfO7vNfRPthR3HPOpWNkRgbbDC3yeXLt,mujeeb_uddin@live.com,Khaja,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,19500,2026-01-14,6,71,6,1,87,0," Neutral  ",1,0,1,SLACK + MEETING,60,70,AVERAGE,0935beb44fe301276400589b94f54be0,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_NPw9sOehmG3RyT70VPmWWjEgpGhaVaPsdv3TM44GcQi,siddhant.navaratna1@gmail.com,Siddhant / Siddhant Navaratna,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,,2024-11-19,915,634,976,,,0,NO_SENTIMENT,,0,,NO DATA,9999,9999,NO DATA,c6dda040158384c0880d68435e2ffb42,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_XdLvff4ZfybY8X3qoAR2C4PYN5VJ8xw00p0k9xQu6Wn,jtcdjs23@gmail.com,Simeon Jones,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,20000,2026-04-01,164,179,163,133,105,0," Positive  ",0,0,0,NO ENGAGEMENT,20,35,POOR,3e0591563f0621cd4b56f2b73af24167,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_SlsTjE2uoy2oI8obdLplJCP8pgO5sGLMaicWIeqdtuU,josueangelsantiago@gmail.com,josueangelsantiago@gmail.com / Josue Angel Santiago,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,,2026-09-19,34,33,45,,,0,NO_SENTIMENT,,0,0,NO DATA,9999,9999,NO DATA,8ce5f3b74e0085d4a6c76cc56fcd5eb3,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_mitMDhSPnOolOaClkxPT8j3BIlLYjuh32Uv0dNupqFS,srinadar@gmail.com,Srinivasa Nadar,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,20000,2026-04-28,51,18,66,8,0,0," Positive  ",1,0,0,SLACK ONLY,30,45,POOR,0eca5f6bafe82a2418e38da7fa66cc89,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_eOWzVQLFV0H4T99WRDtKOnHDpM58L6C7l0N9cxIYvIJ,o_dewan@yahoo.com,Omar Dewan,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,22000,2026-02-23,58,59,76,2,15,0," Positive  ",1,0,0,SLACK ONLY,30,45,POOR,ea402051a841e9eef3975cc8d4b679e4,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_xUaGHejcIKcRKTJNhXtxcZ9rmCxB7PYMLWPzloJhIe4,roy.mendiola@gmail.com,Roy Mendiola,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2024-11-19,671,704,736,219,0,0," Positive  ",0,0,,NO DATA,9999,9999,NO DATA,e7cbd182a837c8c1eb05f6f2b4d540cb,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_Cxdbwk0OOI8UZdSlUSFIhUPLbG5NSSkMUJGTJUhvUfj,pkamalkumar@gmail.com,Kamal Potnuru,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2025-12-17,150,209,11,2,10,0," Positive  ",1,0,1,SLACK + MEETING,60,75,GOOD,f89a4d75f7bd669ab2f3998afafbefae,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_DFUQocqAHFDIA9bFrZbtZLEZXM0aaLAB0gTCQdZo5Yf,pbrousseau74@gmail.com,Peter Rousseau,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,19000,2026-06-28,18,13,18,1,4,0," Neutral  ",1,1,1,SLACK + PLATFORM + MEETING,90,100,GOOD,4bf7a75b9e71eb1dc3af78bccf6bbf7a,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_vSsKQbSrKLkEtonNqv8VBNf5utqPMDs09DnCAVk8aNF,real.tamaratalbott@gmail.com,Tamara Talbott,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,,2024-12-31,159,154,163,106,107,0," Positive  ",0,0,0,NO ENGAGEMENT,20,35,POOR,13b1eb4851240688afb9b8251a90935f,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_dRC1pjtWmAtb57UJ5lGyr7oqEw7xOKLNQytliEsi34l,dialloadro@gmail.com / durrah149@outlook.com,abdourahamane diallo,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,5000,2025-09-04,80,424,74,70,4,0," Positive  ",0,0,0,NO ENGAGEMENT,20,35,POOR,0ef1fe47e02e86e0ae20233d392d788a,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_g4EDJcfm6BqaqNTBYS6zfJM0Ew3vsdKKGgwBIONlj8v,lkrjagadeeshreddy.1995@gmail.com,Jagadeesh,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,11000,2024-11-19,473,10,228,211,20,0," Positive  ",0,1,0,PLATFORM ONLY,50,65,AVERAGE,7a8f61bcd8636e70e0a86f041bb0ca79,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646


In [0]:
import io
import pandas as pd

# Reference output pasted in the cell above, verbatim (VRSN window 2026-11-01 -> 2026-11-02)
teacher_txt = """LEAD_ID,LEAD_EMAIL,LEAD_NAME,VRSN_START_DATE,VRSN_END_DATE,CONTRACT_VALUE,CONTRACT_START_DATE,DAYS_SINCE_LAST_EMAIL,DAYS_SINCE_LAST_LOGIN,DAYS_SINCE_LAST_MEETING,DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT,DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,TOTAL_MISSED_CALLS,SENTIMENTS_LAST_30_DAYS,SLACK_ENGAGED_FLAG,PLATFORM_ENGAGED_FLAG,MEETING_ENGAGED_FLAG,ENGAGEMENT_PATTERN,BASE_ENGAGEMENT_SCORE,FINAL_SCORE,HEALTH_BAND,HASH_VALUE,SOURCE_UPDATED_AT,INSERT_DATE
lead_bo0Aw17YkLn795YW2j2XHTsh43KqkhqjFMjDG1UwH38,sydibuari@yahoo.com / sbuari88@gmail.com,Sydi Buari,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2024-12-26,363,623,370,330,213,0, Positive  ,0,0,,NO DATA,20,9999,NO DATA,c847cd44e3f4f3087bc12d9cd102e173,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_NEAsiuAhEAuKTWYXmB2c9EEcf7wLlSqRSjMKNX3xdGH,oyegunwaseyi@gmail.com,Seyi Oyegunwa,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,20000,2026-01-27,177,142,50,83,110,0, Positive  ,0,0,0,NO ENGAGEMENT,20,35,POOR,45570028e917ff4db7185bf4b00de031,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_C7o1N9SfcGAsLCqiHGxxeOmVoWQN7ShD9LzATOWcdcm,irron.williams@gmail.com,Irron Williams,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,7000,2025-05-30,375,231,343,259,221,0, Positive  ,0,0,,NO DATA,9999,9999,NO DATA,3fb3932d75a87a85dd67bb498ea894bf,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_Pvq800BVsA671hrI7S6hA4fbAD2dUOeWYJTjL4P8e9n,annambasit123@gmail.com,Annam Basit,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2025-01-11,529,330,445,333,12,0, Positive  ,0,0,,NO DATA,9999,9999,NO DATA,e2a9f37036151d4602266be5ffb923bd,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_9ppVFUPLYDFu7DboruWWUaqWDMq7piyhWxS4enw5oxy,suryadevarabasaveswaramukesh@gmail.com,Basaveswara Mukesh Suryadevara,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,8000,2026-06-13,33,14,29,9,0,0, Positive  ,1,1,1,SLACK + PLATFORM + MEETING,90,105,GOOD,49002d77f222db86f78033bb1367632a,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_8kvThfspxJJBRfsNxOYhVCuxsQWSZu8CJEF3wLWCX76,nicholo.neri@gmail.com,Nic Neri,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,16000,2025-11-20,230,135,19,215,21,0, No Student Activity  ,0,0,1,MEETING ONLY,40,45,POOR,dc5eaf6196e4405bbbd9f258b9a3856c,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_DBOhRgpYGb8kIQXypKsKdxxTy97QY0kJ8CBYsmmL4SG,attribution.matters@gmail.com,Don Attribution,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,19000,2025-10-28,26,2,2,1,2,0, Neutral  ,1,1,1,SLACK + PLATFORM + MEETING,90,100,GOOD,10df54f90b7e09c4f409ed683aa5d79a,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_pAq25PJR446RfO7vNfRPthR3HPOpWNkRgbbDC3yeXLt,mujeeb_uddin@live.com,Khaja,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,19500,2026-01-14,6,71,6,1,87,0, Neutral  ,1,0,1,SLACK + MEETING,60,70,AVERAGE,0935beb44fe301276400589b94f54be0,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_NPw9sOehmG3RyT70VPmWWjEgpGhaVaPsdv3TM44GcQi,siddhant.navaratna1@gmail.com,Siddhant Navaratna,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,,2024-11-19,915,634,976,,,0,NO_SENTIMENT,,0,,NO DATA,9999,9999,NO DATA,c6dda040158384c0880d68435e2ffb42,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_XdLvff4ZfybY8X3qoAR2C4PYN5VJ8xw00p0k9xQu6Wn,jtcdjs23@gmail.com,Simeon Jones,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,20000,2026-04-01,164,179,163,133,105,0, Positive  ,0,0,0,NO ENGAGEMENT,20,35,POOR,3e0591563f0621cd4b56f2b73af24167,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_SlsTjE2uoy2oI8obdLplJCP8pgO5sGLMaicWIeqdtuU,josueangelsantiago@gmail.com,Josue Angel Santiago,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,,2026-09-19,34,33,45,,,0,NO_SENTIMENT,,0,0,NO DATA,9999,9999,NO DATA,8ce5f3b74e0085d4a6c76cc56fcd5eb3,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_mitMDhSPnOolOaClkxPT8j3BIlLYjuh32Uv0dNupqFS,srinadar@gmail.com,Srinivasa Nadar,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,20000,2026-04-28,51,18,66,8,0,0, Positive  ,1,0,0,SLACK ONLY,30,45,POOR,0eca5f6bafe82a2418e38da7fa66cc89,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_eOWzVQLFV0H4T99WRDtKOnHDpM58L6C7l0N9cxIYvIJ,o_dewan@yahoo.com,Omar Dewan,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,22000,2026-02-23,58,59,76,2,15,0, Positive  ,1,0,0,SLACK ONLY,30,45,POOR,ea402051a841e9eef3975cc8d4b679e4,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_xUaGHejcIKcRKTJNhXtxtcZ9rmCxB7PYMLWPzloJhIe4,roy.mendiola@gmail.com,Roy Mendiola,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2024-11-19,671,704,736,219,0,0, Positive  ,0,0,,NO DATA,9999,9999,NO DATA,e7cbd182a837c8c1bd05f6f2b4d540cb,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_Cxdbwk0OOI8UZdSlUSFIhUPLbG5NSSkMUJGTJUhvUfj,pkamalkumar@gmail.com,Kamal Potnuru,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,10000,2025-12-17,150,209,11,2,10,0, Positive  ,1,0,1,SLACK + MEETING,60,75,GOOD,f89a4d75f7bd669ab2f3998afafbefae,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_DFUQocqAHFDIA9bFrZbtZLEZXM0aaLAB0gTCQdZo5Yf,pbrousseau74@gmail.com,Peter Rousseau,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,19000,2026-06-28,18,13,18,1,4,0, Neutral  ,1,1,1,SLACK + PLATFORM + MEETING,90,100,GOOD,4bf7a75b9e71eb1dc3af78bccf6bbf7a,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_vSsKQbSrKLkEtonNqv8VBNf5utqPMDs09DnCAVk8aNF,real.tamaratalbott@gmail.com,Tamara Talbott,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,,2024-12-31,159,154,163,106,107,0, Positive  ,0,0,0,NO ENGAGEMENT,20,35,POOR,13b1eb4851240688afb8b825a90935f,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_dRC1pjtWmAtb57UJ5lGyr7oqEw7xOKLNQytliEsi34l,dialloadro@gmail.com / durrah149@outlook.com,abdourahamane diallo,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,5000,2025-09-04,80,424,74,70,4,0, Positive  ,0,0,0,NO ENGAGEMENT,20,35,POOR,0ef1fe47e02e86e0ae20233d392d788a,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646
lead_g4EDJcfm6BqaqNTBYS6zfJM0Ew3vsdKKGgwBIONlj8v,lkrjagadeeshreddy.1995@gmail.com,Jagadeesh,2026-11-01 09:06:21.987,2026-11-02 09:06:35.064,11000,2024-11-19,473,10,228,211,20,0, Positive  ,0,1,0,PLATFORM ONLY,50,65,AVERAGE,7a8f61bcd8636e70e0a86f041bb0ca79,2026-11-01 09:06:21.987,2026-11-01 01:06:47.646"""

t = pd.read_csv(io.StringIO(teacher_txt), skipinitialspace=True)
t['SENTIMENTS_LAST_30_DAYS'] = t['SENTIMENTS_LAST_30_DAYS'].astype(str).str.strip()

# Reference conventions: FINAL_SCORE/BASE 9999 means NO DATA -> our NULL
for c in ['BASE_ENGAGEMENT_SCORE', 'FINAL_SCORE']:
    t[c] = t[c].where(t[c] != 9999, other=pd.NA)

ids = ", ".join(f"'{x}'" for x in t['LEAD_ID'])
ours = spark.sql(f"""
    SELECT LEAD_ID, CONTRACT_START_DATE, DAYS_SINCE_LAST_LOGIN,
           SLACK_ENGAGED_FLAG, PLATFORM_ENGAGED_FLAG, MEETING_ENGAGED_FLAG,
           ENGAGEMENT_PATTERN, BASE_ENGAGEMENT_SCORE, FINAL_SCORE, HEALTH_BAND, VRSN_START_DATE
    FROM crm_ingestion.gold.customer_health_profile
    WHERE VRSN_END_DATE IS NULL AND LEAD_ID IN ({ids})
""").toPandas()

m = ours.merge(t, on='LEAD_ID', how='left', suffixes=('_ours', '_ref'))
missing = t[~t['LEAD_ID'].isin(ours['LEAD_ID'])]
print(f"Compared {len(ours)} of {len(t)} reference leads (missing: {len(missing)})")
if len(missing):
    print("Missing from our population: " + ", ".join(
        f"{r['LEAD_NAME']} ({r['ENGAGEMENT_PATTERN']})" for _, r in missing.iterrows()))

def same(a, b):
    # Numeric-aware: spark floats ('0.0') vs reference ints ('0') must compare equal
    a, b = pd.Series(a), pd.Series(b)
    both_na = a.isna() & b.isna()
    an, bn = pd.to_numeric(a, errors='coerce'), pd.to_numeric(b, errors='coerce')
    num_ok = an.notna() & bn.notna() & (an == bn)
    str_ok = (a.astype(str).str.strip() == b.astype(str).str.strip()) & ~both_na
    return both_na | num_ok | str_ok

# Flags/patterns/scores/bands are drift-free; login date drifts ~4 days (ref snapshot 11-02 vs ours 11-06) -> tolerance 5
for c in ['SLACK_ENGAGED_FLAG', 'PLATFORM_ENGAGED_FLAG', 'MEETING_ENGAGED_FLAG',
          'ENGAGEMENT_PATTERN', 'BASE_ENGAGEMENT_SCORE', 'FINAL_SCORE', 'HEALTH_BAND', 'CONTRACT_START_DATE']:
    m[c + '_match'] = same(m[c + '_ours'], m[c + '_ref'])

log_o, log_r = m['DAYS_SINCE_LAST_LOGIN_ours'], m['DAYS_SINCE_LAST_LOGIN_ref']
m['LOGIN_match'] = ((log_o.isna() & log_r.isna())
                    | ((log_o.notna() & log_r.notna()) & ((log_o - log_r).abs() <= 5)))

compare_cols = ['CONTRACT_START_DATE', 'DAYS_SINCE_LAST_LOGIN', 'SLACK_ENGAGED_FLAG',
                'PLATFORM_ENGAGED_FLAG', 'MEETING_ENGAGED_FLAG', 'ENGAGEMENT_PATTERN',
                'BASE_ENGAGEMENT_SCORE', 'FINAL_SCORE', 'HEALTH_BAND']

print('\n=== MATCH COUNTS (of 19) ===')
for c in compare_cols:
    print(f"{c:26s}: {int(m[c + '_match'].sum()) if c != 'DAYS_SINCE_LAST_LOGIN' else int(m['LOGIN_match'].sum())}")

print('\n=== SIDE-BY-SIDE (key disagreement columns) ===')
side = pd.DataFrame({
    'lead': m['LEAD_ID'].str.slice(5, 15),
    'PATTERN_ours': m['ENGAGEMENT_PATTERN_ours'],
    'PATTERN_ref': m['ENGAGEMENT_PATTERN_ref'],
    'PATTERN_match': m['ENGAGEMENT_PATTERN_match'],
    'PLATFORM_ours': m['PLATFORM_ENGAGED_FLAG_ours'],
    'PLATFORM_ref': m['PLATFORM_ENGAGED_FLAG_ref'],
    'PLATFORM_match': m['PLATFORM_ENGAGED_FLAG_match'],
    'LOGIN_ours': m['DAYS_SINCE_LAST_LOGIN_ours'],
    'LOGIN_ref': m['DAYS_SINCE_LAST_LOGIN_ref'],
    'LOGIN_match': m['LOGIN_match'],
    'CSTART_ours': m['CONTRACT_START_DATE_ours'],
    'CSTART_ref': m['CONTRACT_START_DATE_ref'],
    'CSTART_match': m['CONTRACT_START_DATE_match'],
    'SCORE_ours': m['FINAL_SCORE_ours'],
    'SCORE_ref': m['FINAL_SCORE_ref'],
    'SCORE_match': m['FINAL_SCORE_match'],
    'BAND_match': m['HEALTH_BAND_match'],
})
display(side)

print('\n=== MISMATCH DETAILS ===')
for c in ['ENGAGEMENT_PATTERN', 'PLATFORM_ENGAGED_FLAG', 'CONTRACT_START_DATE', 'FINAL_SCORE', 'HEALTH_BAND']:
    bad = m[~m[c + '_match']]
    if len(bad):
        print(f"\n-- {c} mismatches ({len(bad)}):")
        for _, r in bad.iterrows():
            print(f"  {r['LEAD_ID'][:20]}... ours={r[c + '_ours']}  ref={r[c + '_ref']}")

In [0]:
%sql
-- The 2 reference leads missing from our population: full lead version history
-- in silver. If the LATEST version is not 'Coaching Client', the lead left the
-- client book (lead_dim = current clients only) — check WHEN the status changed
-- relative to the reference snapshot (2026-11-01 -> 2026-11-02).
SELECT
    id AS lead_id,
    status_label,
    date_updated,
    bronze_insert_date,
    ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS version_rank_desc
FROM crm_ingestion.silver.leads_processed
WHERE id IN (
    'lead_C7o1N9SfcGAsLCqiHGxxeOmVoWQN7ShD9LzATOWcdcm',  -- Irron Williams
    'lead_xUaGHejcIKcRKTJNhXtxtcZ9rmCxB7PYMLWPzloJhIe4'   -- Roy Mendiola
)
ORDER BY id, date_updated, bronze_insert_date

In [0]:
%sql
-- Roy Mendiola hunt: does the lead exist ANYWHERE in our lakehouse, and how big
-- is the 'Customer' status population the teacher may be including?
SELECT 'BRONZE leads_raw rows for teacher Roy lead_id' AS check, COUNT(*) AS n
FROM crm_ingestion.bronze.leads_raw
WHERE CAST(raw_data AS STRING) LIKE '%lead_xUaGHejcIKcRKTJNhXtxtcZ9rmCxB7PYMLWPzloJhIe4%'
UNION ALL
SELECT 'GOLD lead_emails leads owning roy.mendiola@gmail.com', COUNT(DISTINCT lead_id)
FROM crm_ingestion.gold.lead_emails
WHERE ARRAY_CONTAINS(customer_emails, 'roy.mendiola@gmail.com')
UNION ALL
SELECT 'SILVER current-version leads with status Customer', COUNT(*)
FROM (
    SELECT id, status_label
    FROM crm_ingestion.silver.leads_processed
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) = 1
)
WHERE status_label = 'Customer'
UNION ALL
SELECT 'SILVER current-version leads with status Coaching Client (sanity)', COUNT(*)
FROM (
    SELECT id, status_label
    FROM crm_ingestion.silver.leads_processed
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) = 1
)
WHERE status_label = 'Coaching Client'